In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

gemeenten = gpd.read_file("../../data/raw/boundaries/wijken_en_gemeenten.gpkg", layer="gemeenten")
amsterdam_grens = gemeenten[gemeenten['gemeentenaam'] == 'Amsterdam']

wijken = gpd.read_file("../../data/raw/boundaries/wijken_en_gemeenten.gpkg", layer="wijken")

amsterdam_wijken = gpd.clip(wijken, amsterdam_grens)

amsterdam_wijken = amsterdam_wijken[~amsterdam_wijken.is_empty]
amsterdam_wijken = amsterdam_wijken[amsterdam_wijken.geometry.notna()]

df = pd.read_csv("../../data/final/df_final_model.csv")

# Koppel
gdf = amsterdam_wijken.merge(
    df,
    left_on='wijknaam',
    right_on='neighbourhood_name',
    how='inner'
)

cols = ['light_density_per_km2', 'tree_density_per_km2', 'mean_dist_nightlife', 'crime_rate_per_1000']
gdf_clean = gdf.dropna(subset=cols).copy()

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Light Density
gdf_clean.plot(
    column='light_density_per_km2',
    cmap='YlOrBr',
    scheme='quantiles',
    k=5,
    legend=True,
    ax=axes[0]
)
axes[0].set_title('Light density (per km²)')
axes[0].axis('off')

# Tree Density
gdf_clean.plot(
    column='tree_density_per_km2',
    cmap='Greens',
    scheme='quantiles',
    k=5,
    legend=True,
    ax=axes[1]
)
axes[1].set_title('Tree density (per km²)')
axes[1].axis('off')

# Facility: Afstand tot Nightlife
gdf_clean.plot(
    column='mean_dist_nightlife',
    cmap='RdPu_r',
    scheme='quantiles',
    k=5,
    legend=True,
    ax=axes[2]
)
axes[2].set_title('Distance to Bars/Nightlife')
axes[2].axis('off')

plt.suptitle("Exploratory Analysis: Design Features", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

# Spatial Weights Matrix
w = Queen.from_dataframe(gdf_clean)
w.transform = 'r'  # Row-standardization

# Global Moran's I voor Crime, Licht en Bomen
vars_to_test = {
    'Crime Rate': 'crime_rate_per_1000',
    'Light Density': 'light_density_per_km2',
    'Tree Density': 'tree_density_per_km2'
}

print("Global Spatial Autocorrelation (Moran's I)")
for name, col in vars_to_test.items():
    moran = Moran(gdf_clean[col], w)
    print(f"\nVariabele: {name}")
    print(f"Moran's I: {moran.I:.3f}")
    print(f"P-value:   {moran.p_sim:.5f}")

In [ ]:
from esda.moran import Moran_Local
from splot.esda import lisa_cluster, plot_local_autocorrelation

# Bereken Local Moran's I voor Crime Rate
y = gdf_clean['crime_rate_per_1000']
lisa = Moran_Local(y, w)

# Plot de Cluster Map
fig, ax = plt.subplots(figsize=(12, 12))
lisa_cluster(lisa, gdf_clean, p=0.05, ax=ax)
ax.set_title("LISA Cluster Map: Crime Hotspots & Coldspots (p < 0.05)")
plt.show()